# Data Science for Business - The lasso on Ames Housing

*Session 5 · ISLP 6.2.2 The Lasso · 6.2.3 Selecting the Tuning Parameter · 6.4 High Dimensions · Lab 6.5.2*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import r2_score, root_mean_squared_error

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_05/ameshousing.csv')

In [ ]:
data.head()

## Prepare data

First, we will remove some columns that are not useful for our task.

In [ ]:
data = data.drop(['YrSold', 'MoSold', 'SaleCondition', 'SaleType'], axis=1)

Next, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets.

In [ ]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Finally, we will do some feature engineering. It is important to use only information from the training set for feature engineering, and then mechanistically repeat these steps on the test set.

Typically, feature engineering depends strongly on the datatype of the variables. Hence, we will first determine which variables are categorical and which are numerical. Subsequently, we will transform these variables separately.

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns

The categorical variables must be transformed into numerical representations, e.g., by one-hot encoding them.

In [ ]:
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
enc.fit(X_train[categorical_features])

X_train_cat = enc.transform(X_train[categorical_features])
X_test_cat = enc.transform(X_test[categorical_features])

X_train_cat = pd.DataFrame(X_train_cat, columns=enc.get_feature_names_out(categorical_features))
X_test_cat = pd.DataFrame(X_test_cat, columns=enc.get_feature_names_out(categorical_features))

In [ ]:
X_train_cat.head()

The numerical variables stay as they are for now. We reset their index so that they line up with the one-hot encoded dataframes when we join them.

In [ ]:
X_train_num = X_train[numerical_features].reset_index(drop=True)
X_test_num = X_test[numerical_features].reset_index(drop=True)

Let's join the engineered categorical and the numerical variables again.

In [ ]:
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

Now we standardize **all** columns (subtract the mean, divide by the standard deviation), including the dummies, as in ISLP Lab 6.5.2. This is especially important for the lasso: the penalty treats all coefficients alike, so they need to be on comparable scales. The scaler is fit on the training data only.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
X_train.head()

In [ ]:
X_train.shape

## The lasso

The lasso minimises $\text{RSS} + \lambda \sum_j |\beta_j|$ (ISLP 6.2.2, eq. 6.7). A larger λ shrinks the coefficients more and sets some of them exactly to zero: automatic feature selection.

sklearn calls λ `alpha` and scales the RSS by $\frac{1}{2n}$: `Lasso` minimises $\frac{1}{2n}\text{RSS} + \alpha \sum_j |\beta_j|$. Multiplying by $2n$ shows that $\alpha = \lambda / (2n)$, so sklearn's alpha values are much smaller than the book's λ values for the same model.

We start with an arbitrary alpha value and fit the model on the training data.

In [ ]:
lasso_mod = Lasso(alpha=100)
lasso_mod.fit(X_train, y_train)

Evaluate the model on both training and test set using *R2* and *RMSE* as metrics.

In [ ]:
# Training data
pred_train = lasso_mod.predict(X_train)
r2_train = r2_score(y_train, pred_train)
rmse_train = root_mean_squared_error(y_train, pred_train)
print('R2 on training set:', round(r2_train, 2))
print('RMSE on training set:', round(rmse_train, 2))

print("===")

# Test data
pred_test = lasso_mod.predict(X_test)
r2_test = r2_score(y_test, pred_test)
rmse_test = root_mean_squared_error(y_test, pred_test)
print('R2 on test set:', round(r2_test, 2))
print('RMSE on test set:', round(rmse_test, 2))

How many coefficients are not zero?

In [ ]:
print((lasso_mod.coef_ != 0).sum(), 'of', X_train.shape[1], 'features kept')

### The coefficient path

Next, we try 100 alpha values between $10^2$ and $10^5$, evenly spaced on a log scale (like the λ grids in ISLP). The loop below fits one lasso model per alpha and collects the estimated coefficients and, for illustration only (see below), the test set RMSE in a dataframe.

In [ ]:
alphas = np.logspace(2, 5, 100)

lasso_mod = Lasso()

results = []
for a in alphas:
    result = {}
    lasso_mod.set_params(alpha=a)
    lasso_mod.fit(X_train, y_train)
    pred_test = lasso_mod.predict(X_test)

    result["alpha"] = a
    result["rmse"] = root_mean_squared_error(y_test, pred_test)   # for illustration only, see below
    for name, value in zip(lasso_mod.feature_names_in_, lasso_mod.coef_):
        result[name] = value

    results.append(result)

In [ ]:
results_df = pd.DataFrame(results)
results_df.head()

Let's visualize how the coefficients shrink with increasing alpha. First, we wrangle the data into long format (one row per alpha and feature).

In [ ]:
results_df_long = results_df.melt(id_vars=['alpha', 'rmse'], var_name='feature', value_name='coefficient')
results_df_long.head()

In [ ]:
sns.lineplot(data=results_df_long, x='alpha', y='coefficient', hue='feature', legend=False)
plt.xscale('log')
plt.xlabel('alpha (log scale)')
plt.ylabel('Standardized coefficients')
plt.show()

Each line is one coefficient. Moving right, they shrink towards zero and drop out one by one; the survivors are the strongest predictors (cf. ISLP Fig. 6.6).

### Illustration only: why not pick alpha on the test set?

Similarly, we can plot the test set RMSE against alpha. It is tempting to simply pick the alpha with the lowest test RMSE. But then the test set is used to *choose* the model, it becomes a validation set, and the RMSE of the chosen model is no longer an honest estimate of its performance on new data. We therefore do **not** tune alpha this way.

In [ ]:
sns.lineplot(data=results_df, x='alpha', y='rmse')
plt.xscale('log')
plt.xlabel('alpha (log scale)')
plt.ylabel('Test RMSE')
plt.title('Illustration only: do not tune on the test set')
plt.show()

## Hyperparameter tuning

The proper way to choose alpha is k-fold cross-validation on the **training** set (ISLP 6.2.3). `LassoCV` fits the lasso for every alpha on each fold and picks the alpha with the lowest average CV error. We shuffle the folds with a fixed seed so that the result is reproducible.

In [ ]:
lasso_mod_cv = LassoCV(alphas=alphas, cv=KFold(5, shuffle=True, random_state=42))
lasso_mod_cv.fit(X_train, y_train)

Which alpha value leads to the best out-of-sample predictive accuracy?

In [ ]:
lasso_mod_cv.alpha_

Plot the CV RMSE against alpha (cf. ISLP Fig. 6.13). `mse_path_` holds the MSE for each alpha (rows) and fold (columns); the dashed line marks the chosen alpha.

In [ ]:
cv_rmse = np.sqrt(lasso_mod_cv.mse_path_.mean(axis=1))

plt.plot(lasso_mod_cv.alphas_, cv_rmse)
plt.axvline(lasso_mod_cv.alpha_, color='black', ls='--')
plt.xscale('log')
plt.xlabel('alpha (log scale)')
plt.ylabel('CV RMSE')
plt.show()

Refit the model with the best alpha value on the full training data (`LassoCV` already does this internally; we do it explicitly here).

In [ ]:
lasso_mod_tuned = Lasso(alpha=lasso_mod_cv.alpha_)
lasso_mod_tuned.fit(X_train, y_train)

How many features survived?

In [ ]:
print((lasso_mod_tuned.coef_ != 0).sum(), 'of', X_train.shape[1], 'features kept')

Evaluate this model on the test set.

In [ ]:
# Training data
pred_train = lasso_mod_tuned.predict(X_train)
r2_train = r2_score(y_train, pred_train)
rmse_train = root_mean_squared_error(y_train, pred_train)
print('R2 on training set:', round(r2_train, 2))
print('RMSE on training set:', round(rmse_train, 2))

print("===")

# Test data
pred_test = lasso_mod_tuned.predict(X_test)
r2_test = r2_score(y_test, pred_test)
rmse_test = root_mean_squared_error(y_test, pred_test)
print('R2 on test set:', round(r2_test, 2))
print('RMSE on test set:', round(rmse_test, 2))

Don't be surprised if this test RMSE is a bit higher than the minimum of the test curve in the illustration above: that minimum was optimistic by construction, because it was picked by looking at the test set. The value here is an honest estimate.

## A secret weapon? The lasso with many interaction terms

The automatic feature selection of the lasso can be used to search for important interaction terms. In the following, we add all two-way interactions between the numerical predictors to the feature matrix, use cross-validation to choose alpha, and then refit and evaluate a final model.

(Interactions with all ~290 columns, including the dummies, would give ~41,000 features, which needs lots of memory and minutes of computing time. We therefore restrict the interactions to the numerical predictors.)

Check the dimensions of the numerical predictors.

In [ ]:
X_train_num.shape

Use the *PolynomialFeatures* transformer to create all pairwise products of the numerical predictors (`interaction_only=True` skips squared terms, `include_bias=False` skips the column of ones).

In [ ]:
interact = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
interact.fit(X_train_num)

X_train_interact = pd.DataFrame(interact.transform(X_train_num), columns=interact.get_feature_names_out())
X_test_interact = pd.DataFrame(interact.transform(X_test_num), columns=interact.get_feature_names_out())

In [ ]:
X_train_interact.shape

Join the interactions with the dummies and standardize all columns again (fit on training data only).

In [ ]:
X_train_interact = pd.concat([X_train_interact, X_train_cat], axis=1)
X_test_interact = pd.concat([X_test_interact, X_test_cat], axis=1)

scaler_interact = StandardScaler()
scaler_interact.fit(X_train_interact)

X_train_interact = pd.DataFrame(scaler_interact.transform(X_train_interact), columns=X_train_interact.columns)
X_test_interact = pd.DataFrame(scaler_interact.transform(X_test_interact), columns=X_test_interact.columns)

Check the dimensions of *X_train_interact* and have a look at the feature matrix.

In [ ]:
X_train_interact.shape

In [ ]:
X_train_interact.head()

With this many features relative to ~2,300 training houses, least squares would overfit badly (ISLP 6.4). The lasso keeps only a few of them. As before, we use k-fold cross-validation to search for the best alpha value and then refit the model with this parameter. The next cell takes a few seconds.

In [ ]:
lasso_mod_interact_cv = LassoCV(alphas=alphas, cv=KFold(5, shuffle=True, random_state=42))
lasso_mod_interact_cv.fit(X_train_interact, y_train)

In [ ]:
lasso_mod_interact_tuned = Lasso(alpha=lasso_mod_interact_cv.alpha_)
lasso_mod_interact_tuned.fit(X_train_interact, y_train)

print('alpha:', lasso_mod_interact_cv.alpha_)
print((lasso_mod_interact_tuned.coef_ != 0).sum(), 'of', X_train_interact.shape[1], 'features kept')

Which features have the largest (absolute) coefficients?

In [ ]:
coefs = pd.Series(lasso_mod_interact_tuned.coef_, index=X_train_interact.columns)
coefs[coefs.abs().sort_values(ascending=False).index].head(10)

Interpret these with care: when there are many correlated candidate features, the lasso picks *one* of many sets that predict about equally well (ISLP 6.4.4). The selected interactions are not "the" true drivers of price.

Before looking at the test set, compare both lasso models by their CV RMSE on the training data (the fair comparison).

In [ ]:
print('CV RMSE without interactions:', round(np.sqrt(lasso_mod_cv.mse_path_.mean(axis=1).min()), 2))
print('CV RMSE with interactions:   ', round(np.sqrt(lasso_mod_interact_cv.mse_path_.mean(axis=1).min()), 2))

Evaluate the tuned lasso model with interactions on the test set.

In [ ]:
# Training data
pred_train = lasso_mod_interact_tuned.predict(X_train_interact)
r2_train = r2_score(y_train, pred_train)
rmse_train = root_mean_squared_error(y_train, pred_train)
print('R2 on training set:', round(r2_train, 2))
print('RMSE on training set:', round(rmse_train, 2))

print("===")

# Test data
pred_test = lasso_mod_interact_tuned.predict(X_test_interact)
r2_test = r2_score(y_test, pred_test)
rmse_test = root_mean_squared_error(y_test, pred_test)
print('R2 on test set:', round(r2_test, 2))
print('RMSE on test set:', round(rmse_test, 2))

So, is it a secret weapon? Not here: the interaction model fits the training data much better, but its CV RMSE is *higher* than without interactions, and the test set confirms it. Products of predictors can take extreme values for a few unusual houses (look at terms like `MiscVal` in the list above), and the model then extrapolates badly. More flexibility is not automatically better (bias-variance trade-off), and CV on the training data told us so before we touched the test set. On other data sets, the same recipe can pay off; always check with CV.

## Your turn!

**Your turn!** Ridge regression (ISLP 6.2.1) shrinks coefficients towards zero but never sets them exactly to zero.

1. Fit a `RidgeCV` model (from `sklearn.linear_model`) on `X_train` with `alphas=np.logspace(-2, 4, 100)` and 5-fold CV. Note: sklearn's `Ridge` minimises $\text{RSS} + \alpha \sum_j \beta_j^2$, so here alpha *is* the book's λ.
2. Compare its test RMSE and the number of non-zero coefficients with the tuned lasso model. Which model would you prefer, and why?

In [ ]:
# YOUR CODE HERE